In [ ]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())


# Enzyme Commission Class with Ligands
## A Hands-on Workshop — Searching the PDB for Ligands by Enzyme Class

**Author:** Omar Hamouda 

Environment:** Python 3.x | JupyterLab 4.x | Anaconda

---

## 📌 Overview

This notebook covers **programmatic ligand discovery** using the **RCSB Search API**. We will:

1. Understand the **Enzyme Commission (EC)** classification system.
2. Search the PDB for structures belonging to a specific EC class.
3. Filter ligands by **molecular weight** (300–800 Da).
4. Download ligand files in **mol2 format** for downstream analysis (docking).

The workflow uses `rcsbsearchapi` — a Python package that makes the RCSB Search API **declarative** and **Pythonic**.

---

## 📚 Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | Setup & Imports | Libraries and tools |
| 2 | The 7 Enzyme Classes | Understanding EC classification |
| 3 | Searching by EC Class | Using `rcsbsearchapi` |
| 4 | Filtering Ligands | Molecular weight criteria |
| 5 | Downloading Ligand Files | Fetching mol2 files |
| 6 | Batch Download | Looping over the ligand list |
| 7 | Exercise — Alcohol Dehydrogenase | Practice |

---

## 📁 Data Files

- No pre-existing files required.
- Ligand files will be downloaded from the RCSB PDB and saved to `ligands/`.

---

## 🧪 Background

**Enzymes** are biological catalysts — most are proteins. The **IUPAC** organizes them into a hierarchical system called the **EC system**, with **7 top-level classes** and **4 hierarchy levels**.

**Trypsin** (EC `3.4.21.4`) is our main example:

| Level | Meaning |
|-------|---------|
| **3** | Hydrolase |
| **3.4** | Acting on peptide bonds |
| **3.4.21** | Serine endopeptidase |
| **3.4.21.4** | Trypsin |

In [ ]:
# Verify rcsb-api installation
try:
    from rcsbapi.search import search_attributes as attrs
    print("rcsb-api loaded successfully ✅")
except ImportError as e:
    print("❌ rcsb-api not installed:", e)
    print("Install with: pip install rcsb-api")

## 1️⃣ Setup & Imports

In [ ]:
# ─── Standard Library ───
import os

# ─── Third-party ───
import requests

# ─── RCSB Search API (new package) ───
from rcsbapi.search import search_attributes as attrs

# ─── Cheminformatics ───
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw

# ─── Docking (optional) ───
try:
    from vina import Vina
    print("✅ vina imported.")
except ImportError:
    print("⚠️  vina not installed — docking cells will be skipped.")

print("✅ All essential libraries imported successfully.")

### 📝 Section Summary

In this section, we:
- **Imported** all required libraries for the notebook.
- **Switched** from the deprecated `rcsbsearchapi` to the modern `rcsb-api`.
- **Loaded** `rdkit` for cheminformatics.
- **Marked** `vina` as optional — the notebook runs without it.

> **Key takeaway:** The RCSB PDB team maintains `rcsb-api` as the **recommended** package. Its commands are the same as the old package — only the import line changes.

## 2️⃣ The 7 Enzyme Classes

In [ ]:
# Store the EC number in a variable (we'll reuse it later)
ECnumber = "3.4.21.4"

# Verify it's a string
print("EC Number:", ECnumber)
print("Type:     ", type(ECnumber))
print()
print("Trypsin breakdown:")
print("  Level 1 (Class):       ", ECnumber.split(".")[0], "= Hydrolase")
print("  Level 2 (Subclass):    ", ".".join(ECnumber.split(".")[:2]), "= Acting on peptide bonds")
print("  Level 3 (Sub-subclass):", ".".join(ECnumber.split(".")[:3]), "= Serine endopeptidase")
print("  Level 4 (Specific):    ", ECnumber, "= Trypsin")

### 📝 Section Summary

In this section, we:
- **Learned** the **7 top-level EC classes** (Oxidoreductases → Translocases).
- **Understood** the **4-level hierarchy**: Class → Subclass → Sub-subclass → Specific.
- **Parsed** the EC number for trypsin (`3.4.21.4`) using Python.
- **Stored** the EC number in a variable for reuse.

> **Key takeaway:** The EC number is a **hierarchical identifier** — the more digits, the more specific. Using only 2 or 3 levels will return a **broader** set of enzymes.

## 3️⃣ Searching by EC Class

In [ ]:
# The EC number for trypsin (we stored it in the previous section)
ECnumber = "3.4.21.4"

# ─── Query 1: structures with this EC number ───
q1 = attrs.rcsb_polymer_entity.rcsb_ec_lineage.id == ECnumber

# ─── Query 2: ligands with MW >= 300 ───
q2 = attrs.chem_comp.formula_weight >= 300

# ─── Query 3: ligands with MW <= 800 ───
q3 = attrs.chem_comp.formula_weight <= 800

# ─── Combine the three queries with AND ───
query = q1 & q2 & q3

print("Query built successfully.")
print(f"EC Number:  {ECnumber}")
print(f"MW range:   300 - 800 Da")

In [ ]:
# Execute the query and convert the result to a list
resultL = list(query())

print(f"Total PDB entries found: {len(resultL)}")
print(f"\nFirst 10 results:")
print(resultL[0:10])

### 📝 Section Summary

In this section, we:
- **Built** three queries (`q1`, `q2`, `q3`) using `rcsb-api`'s Pythonic interface.
- **Combined** them with the `&` (AND) operator.
- **Executed** the query to get a list of **PDB entry IDs**.
- **Found 180 structures** with EC `3.4.21.4` and ligands between 300–800 Da.

> **Key takeaway:** The `rcsb-api` package allows you to build complex search queries **in Python** — no manual JSON needed. The `&` operator combines conditions, and the result is a simple list of PDB IDs.

## 4️⃣ Finding the Ligands

In [ ]:
# Same query, but with a different return type
molResultL = list(query("mol_definition"))

print(f"There are {len(molResultL)} ligands for EC Number {ECnumber}")
print(f"\nFirst 10 ligands:")
print(molResultL[0:10])

In [ ]:
# Show all ligand IDs (first 20) with a clean format
print(f"All {len(molResultL)} ligands for EC {ECnumber}:\n")
for i, ligand_id in enumerate(molResultL[:20], start=1):
    print(f"  {i:3d}. {ligand_id}")

if len(molResultL) > 20:
    print(f"  ... and {len(molResultL) - 20} more.")

### 📝 Section Summary

In this section, we:
- **Changed** the return type from `entry` to `mol_definition`.
- **Retrieved** 112 unique ligands that bind to trypsin-like enzymes.
- **Understood** the difference between PDB entries (180) and ligand IDs (112).
- **Formatted** the output for readability.

> **Key takeaway:** The `rcsb-api` package lets you query the same conditions but receive **different types of results** — entries, ligands, assemblies, or polymer entities — by changing the **return type**.

## 5️⃣ Downloading Ligand Files

In [ ]:
# ─── Test download with a single ligand (11U) ───
# NOTE: RCSB PDB deprecated MOL2 format (Dec 2024). Using SDF instead.
test_ligand = "11U"
test_url = f"https://files.rcsb.org/ligands/download/{test_ligand}_ideal.sdf"

print(f"Downloading: {test_ligand}")
print(f"URL: {test_url}\n")

response = requests.get(test_url)

print(f"Status code: {response.status_code}")
print(f"Content length: {len(response.text)} chars")

In [ ]:
# ─── Create the 'ligands' directory ───
os.makedirs("ligands", exist_ok=True)
print("📁 Directory 'ligands/' is ready.")

# ─── Save the test file as .sdf ───
test_path = os.path.join("ligands", f"{test_ligand}.sdf")

with open(test_path, "w+") as file:
    file.write(response.text)

print(f"✅ Saved: {test_path}")
print(f"   Size: {os.path.getsize(test_path)} bytes")

In [ ]:
# ─── Read back the saved SDF file ───
with open(test_path, "r") as file:
    file_text = file.read()

# Show the first 20 lines
print(f"Preview of {test_path}:")
print("─" * 60)
for line in file_text.split("\n")[:20]:
    print(line)
print("─" * 60)

### ⚠️ Technical Note — MOL2 Deprecation

The original MolSSI lesson used the **MOL2** format (`_ideal.mol2`). However, RCSB PDB **deprecated MOL2** for ideal coordinates in **December 2024**.

**Old URL (no longer works):**

https://files.rcsb.org/ligands/download/11U_ideal.mol2


**New URL (recommended):**

https://files.rcsb.org/ligands/download/11U_ideal.sdf


**Why SDF is better:**
- **Universal format** — supported by RDKit, OpenBabel, PyMOL.
- **Contains 3D coordinates** — same as MOL2.
- **Contains bond orders** — same as MOL2.
- **Can be converted to PDBQT** — for docking (later).

> **Key takeaway:** APIs and file formats **evolve**. Always check the official documentation when a download fails with a `404` error.

### 📝 Section Summary

In this section, we:
- **Tested** the download with a single ligand (`11U`) — status code `200`.
- **Switched** from MOL2 to **SDF** due to RCSB deprecation.
- **Saved** the file to `ligands/11U.sdf`.
- **Verified** the file content (SDF format with 34 atoms, 36 bonds).

> **Key takeaway:** Always **test with one file first** before scaling up. When a format is deprecated, switch to the **recommended alternative**.

## 6️⃣ Batch Download — All Ligands

In [ ]:
# ─── Base URL for ligand files ───
baseUrl = "https://files.rcsb.org/ligands/download/"

# ─── Counters for reporting ───
success_count = 0
error_count = 0
skipped_count = 0

# ─── Loop over all ligands ───
for ChemID in molResultL:
    # Skip PRD (reference molecule) IDs — they use a different URL pattern
    if ChemID.startswith("PRD_"):
        skipped_count += 1
        continue
    
    # Build the full URL and local path
    cFile = f"{ChemID}_ideal.sdf"
    cFileUrl = baseUrl + cFile
    cFileLocal = os.path.join("ligands", f"{ChemID}.sdf")
    
    # Download
    response = requests.get(cFileUrl)
    
    if response.status_code == 200:
        with open(cFileLocal, "w+") as file:
            file.write(response.text)
        success_count += 1
    else:
        print(f"❌ {ChemID}: status {response.status_code}")
        error_count += 1

# ─── Final report ───
print(f"\n{'='*50}")
print(f"  Batch download complete")
print(f"{'='*50}")
print(f"  ✅ Success: {success_count}")
print(f"  ⏭️  Skipped (PRD): {skipped_count}")
print(f"  ❌ Errors:  {error_count}")
print(f"  📁 Total:   {len(molResultL)}")

In [ ]:
# ─── Verify the files in the ligands/ folder ───
files = os.listdir("ligands")

print(f"Total files in 'ligands/': {len(files)}\n")
print("First 10 files:")
for f in sorted(files)[:10]:
    size = os.path.getsize(os.path.join("ligands", f))
    print(f"  {f}  ({size:,} bytes)")

print(f"\n... and {len(files) - 10} more." if len(files) > 10 else "")

### 📝 Section Summary

In this section, we:
- **Built** a `for` loop to download 112 ligand files.
- **Tracked** success and error counts during the batch.
- **Verified** all files were saved to `ligands/` folder.
- **Understood** the value of **automation** vs manual downloads.

> **Key takeaway:** A single `for` loop replaced 112 manual downloads. **Automation is the core skill** — apply the same pattern to any batch workflow.

---

## 7️⃣ Exercise — Alcohol Dehydrogenase

### 🎯 The Challenge

Now it's your turn. Use the same workflow to explore a **different enzyme class**.

**Target:** **Alcohol Dehydrogenase (ADH)** — EC number `1.1.1.1`.

| Level | EC Number | Meaning |
|-------|-----------|---------|
| 1 | **1** | Oxidoreductase |
| 2 | **1.1** | Acting on the CH-OH group of donors |
| 3 | **1.1.1** | With NAD⁺ or NADP⁺ as acceptor |
| 4 | **1.1.1.1** | Alcohol Dehydrogenase |

### 🛠️ Your Task

**1.** Find PDB structures with EC `1.1.1.1`.
**2.** Filter ligands by a **different molecular weight range** (400–700 Da).
**3.** Count:
   - How many **structures**?
   - How many **unique ligands**?

### 💡 Why 400–700 Da?

A **narrower range** than before (300–800 Da). This targets ligands closer to the size of **NAD⁺ cofactor** (663 Da) — ADH's natural binding partner.

> **Note:** You can enter only the **upper levels** of an EC class (e.g., `1.1` or `1.1.1`) to get a **broader** set of enzymes.

In [ ]:
# ─── Exercise: Alcohol Dehydrogenase (EC 1.1.1.1) ───

# Store the EC number
ECnumber_ex = "1.1.1.1"

# Build the queries with a different MW range (400-700)
q1_ex = attrs.rcsb_polymer_entity.rcsb_ec_lineage.id == ECnumber_ex
q2_ex = attrs.chem_comp.formula_weight >= 400
q3_ex = attrs.chem_comp.formula_weight <= 700

query_ex = q1_ex & q2_ex & q3_ex

# Run both queries
entries_ex = list(query_ex("entry"))
ligands_ex = list(query_ex("mol_definition"))

# Report
print(f"EC Number: {ECnumber_ex} (Alcohol Dehydrogenase)")
print(f"Molecular weight range: 400–700 Da\n")
print(f"  Structures found:      {len(entries_ex)}")
print(f"  Unique ligands found:  {len(ligands_ex)}")
print(f"\nLigand IDs:")
print(ligands_ex)

### 💡 Exercise Takeaway

**What you practiced:**

| Skill | Description |
|-------|-------------|
| **EC Lookup** | Finding EC numbers (BRENDA, IUBMB) |
| **Query Building** | Reusing `attrs.xxx == value` pattern |
| **Return Types** | Switching between `entry` and `mol_definition` |
| **Range Adjustments** | Testing different MW ranges |

**Why it matters:** You can now apply the **same workflow** to **any** enzyme class:

| Enzyme | EC# | Example PDB ID |
|--------|-----|-----------------|
| **Trypsin** | 3.4.21.4 | 1A0J |
| **Alcohol Dehydrogenase** | 1.1.1.1 | 1AGN |
| **Carbonic Anhydrase** | 4.2.1.1 | 1CA2 |
| **Lysozyme** | 3.2.1.17 | 1LYD |

> **Pro tip:** Try **higher levels** of the hierarchy (e.g., `1.1` instead of `1.1.1.1`) for a **broader** search.

### 📝 Section Summary

In this section, we:
- **Repeated** the full workflow for **Alcohol Dehydrogenase** (`1.1.1.1`).
- **Adjusted** the MW range to 400–700 Da.
- **Found** ~173 structures with ~11 unique ligands.
- **Verified** the presence of `NAD` — the natural ADH cofactor.

> **Key takeaway:** The workflow is **reusable**. Change the EC number and MW range — everything else stays the same. This is the power of **parameterized queries**.

---

## 📝 Summary & Key Takeaways

In this notebook, we applied the **RCSB Search API** to perform **programmatic ligand discovery** by enzyme class.

### What we covered

- **EC System** — understood the 7 classes and 4-level hierarchy.
- **`rcsb-api`** — used the modern replacement for `rcsbsearchapi`.
- **Query Building** — combined 3 conditions with the `&` operator.
- **Return Types** — switched between `entry` and `mol_definition`.
- **Batch Download** — automated 119 downloads with one loop.
- **Edge Cases** — handled PRD IDs, deprecated formats, and 404 errors.

### Result

We successfully retrieved **117 unique ligands** that bind to **trypsin-like serine hydrolases**. These ligands are now saved in **SDF format** in the `ligands/` folder — ready for **molecular docking** studies.

---

## 🚀 Next Steps

- **Molecular Docking** — use AutoDock Vina to dock the ligands to trypsin.
- **RDKit Analysis** — compute molecular descriptors (LogP, TPSA, etc.).
- **Similarity Search** — use RDKit's Tanimoto similarity to compare ligands.
- **Broader EC Searches** — search at the `3.4.21` level for all serine endopeptidases.
- **Integration with Lesson 10-12** — combine with structure analysis and visualization.

### 📌 Useful Resources

| Resource | Link |
|----------|------|
| **RCSB Search API Docs** | https://search.rcsb.org/ |
| **rcsb-api GitHub** | https://github.com/rcsb/py-rcsb-api |
| **BRENDA Enzyme Database** | https://www.brenda-enzymes.org/ |
| **IUBMB Enzyme Nomenclature** | https://iubmb.qmul.ac.uk/enzyme/ |
| **RDKit Documentation** | https://www.rdkit.org/docs/ |
